In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

URL = "https://www.healthline.com/nutrition/foods-to-avoid-with-kidney-disease#foods-to-avoid"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

page_title = soup.title.get_text(" ", strip=True) if soup.title else ""
print("HTTP status:", response.status_code)
print("Page title:", page_title)
print("Downloaded HTML size:", len(response.text))

HTTP status: 200
Page title: Kidney Disease: 17 Foods to Avoid
Downloaded HTML size: 671360


In [2]:
title_match = re.search(r"(\d+)\s+Foods?\s+to\s+Avoid", page_title, flags=re.I)

if title_match is None:
    h1 = soup.find("h1")
    h1_text = h1.get_text(" ", strip=True) if h1 else ""
    title_match = re.search(r"(\d+)\s+Foods?\s+to\s+Avoid", h1_text, flags=re.I)

if title_match is None:
    raise RuntimeError(
        "Could not determine the number of food items from the article title. "
        "The site may have returned a block page or changed its title format."
    )

expected_count = int(title_match.group(1))
print("Expected numbered food headings:", expected_count)

Expected numbered food headings: 17


In [3]:
article = soup.find("article") or soup.find("main") or soup

heading_pattern = re.compile(r"^\s*(\d{1,2})\s*[.)]\s*(.+?)\s*$")
items_by_number = {}

for heading in article.find_all(["h1", "h2", "h3", "h4", "h5", "h6"]):
    heading_text = " ".join(heading.get_text(" ", strip=True).split())
    match = heading_pattern.match(heading_text)

    if match:
        number = int(match.group(1))
        name = match.group(2).strip()

        if 1 <= number <= expected_count:
            items_by_number.setdefault(number, name)

if len(items_by_number) < expected_count:
    for element in article.find_all(["div", "p", "span"]):
        text = " ".join(element.get_text(" ", strip=True).split())

        if len(text) > 120 or len(element.find_all(["p", "div"])) > 0:
            continue

        match = heading_pattern.match(text)
        if match:
            number = int(match.group(1))
            name = match.group(2).strip()
            if 1 <= number <= expected_count:
                items_by_number.setdefault(number, name)

food_items = [
    items_by_number[number]
    for number in range(1, expected_count + 1)
    if number in items_by_number
]

print(f"Extracted {len(food_items)} of {expected_count} food headings:")
for index, food in enumerate(food_items, start=1):
    print(f"{index}. {food}")

missing_numbers = [
    number for number in range(1, expected_count + 1)
    if number not in items_by_number
]

if missing_numbers:
    raise RuntimeError(
        f"Could not extract all numbered headings. Missing item numbers: {missing_numbers}. "
        "Inspect the downloaded HTML"
    )

Extracted 17 of 17 food headings:
1. Dark-colored soda
2. Avocados
3. Canned foods
4. Whole wheat bread
5. Brown rice
6. Bananas
7. Dairy
8. Oranges and orange juice
9. Processed meats
10. Pickles, olives, and relish
11. Apricots
12. Potatoes and sweet potatoes
13. Tomatoes
14. Packaged, instant, and premade meals
15. Swiss chard, spinach, and beet greens
16. Dates, raisins, and prunes
17. Pretzels, chips, and crackers


In [4]:
df = pd.DataFrame({"food_item": food_items})

output_file = "kidney_disease_foods_to_avoid.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} food items to {output_file}")
display(df)

Saved 17 food items to kidney_disease_foods_to_avoid.csv


,food_item
0,Dark-colored soda
1,Avocados
2,Canned foods
3,Whole wheat bread
4,Brown rice
5,Bananas
6,Dairy
7,Oranges and orange juice
8,Processed meats
9,"Pickles, olives, and relish"
